# Bedrock Module · Day 18 — **Capstone: Financial RAG Agent**

**Phase 1 · Module 2: AWS Bedrock & AgentCore** · ~3 hrs

Seven days of parts, one machine today. Build a **Loan Policy Q&A Agent** on the Bedrock stack: a Knowledge Base retrieves policy passages, a Bedrock model answers **grounded only in what was retrieved**, a Guardrail vets both the question and the answer, and the result comes back as a **typed, cited** record. Then we **evaluate** it — 10 test questions, and measure the number that matters to a regulator: the **hallucination rate**.

It runs as a **real LangGraph `StateGraph`** — `guard-in → retrieve → generate → guard-out` — with each Bedrock service standing in as a keyless `Fake*` that mirrors the real API. The closing table maps every fake to the real call: production is a handful of one-line swaps.

**What today reuses**
| From | Reused as |
|---|---|
| Day 12 Knowledge Bases | `FakeKnowledgeBase.retrieve` — IDF-scored top-k chunks |
| Day 11/17 Bedrock model | `FakeBedrockRuntime.converse` — grounded generation |
| Day 16 Guardrails | `FakeGuardrail.apply` — denied topics on input, PII on output |
| **This morning** | `JsonOutputParser` — answer → typed `{answer, citations, confidence}` |
| Day 17 LangGraph | real `StateGraph` + **conditional routing** |


## 0. The corpus — mock loan-policy "PDFs"

A real Knowledge Base ingests PDFs from S3, chunks and embeds them into a vector store (OpenSearch). Here the corpus is a list of **policy chunks**, each with a document id (the citation handle) and text. Twenty short chunks stand in for twenty policy PDFs.

In [1]:
POLICY_CORPUS = [
    ("POL-001", "Residential mortgages require a minimum deposit of 5% of the property value."),
    ("POL-002", "The maximum loan-to-value (LTV) for a residential mortgage is 95%."),
    ("POL-003", "Buy-to-let mortgages require a minimum deposit of 25%; the maximum LTV is 75%."),
    ("POL-004", "Applicants must be at least 18 years old to hold a personal loan."),
    ("POL-005", "Personal loans range from 1,000 to 50,000 pounds over 1 to 7 years."),
    ("POL-006", "The debt-to-income (DTI) ratio must not exceed 45% for loan approval."),
    ("POL-007", "A credit score below 580 results in automatic referral to manual underwriting."),
    ("POL-008", "Early repayment of a personal loan incurs no penalty after the first 12 months."),
    ("POL-009", "Overdraft facilities are capped at 2,000 pounds for standard current accounts."),
    ("POL-010", "Mortgage applications require two years of verified income documentation."),
    ("POL-011", "Self-employed applicants must provide three years of certified accounts."),
    ("POL-012", "The standard variable rate (SVR) applies after any fixed-rate period ends."),
    ("POL-013", "First-time buyers may qualify for a 40-year maximum mortgage term."),
    ("POL-014", "Interest-only mortgages require a documented, credible repayment strategy."),
    ("POL-015", "Joint applications assess the combined income of up to four applicants."),
    ("POL-016", "A missed payment is reported to credit bureaus after 30 days in arrears."),
    ("POL-017", "Business loans require a minimum trading history of 24 months."),
    ("POL-018", "Secured loans use the property as collateral; the bank may repossess on default."),
    ("POL-019", "Payment holidays of up to 3 months are available once per 12-month period."),
    ("POL-020", "Foreign-currency income is discounted by 10% when assessing affordability."),
]
print(len(POLICY_CORPUS), "policy chunks loaded (stand-in for 20 ingested PDFs)")

20 policy chunks loaded (stand-in for 20 ingested PDFs)


☝ Each tuple is `(doc_id, text)` — `doc_id` is what the agent must **cite**. In real Bedrock these are chunks the Knowledge Base created from your S3 PDFs; the `doc_id` becomes the citation's `location`.

## 1. `FakeKnowledgeBase` — the retrieval half of RAG

Real Bedrock embeds the query and does vector search. Keyless, we score by **IDF-weighted term overlap** — rare, meaningful terms (`buy-to-let`, `overdraft`, `18`) count far more than common ones (`the`, `loan`, `maximum`). That's a transparent stand-in for cosine similarity (you build the real thing from scratch on Day 20). A **relevance threshold** means an off-topic question retrieves *nothing* — the first line of defence against hallucination.

In [2]:
import re, math
from collections import Counter

STOP = {"the","a","an","of","for","to","is","are","and","or","in","on","at","my","i",
        "what","how","much","many","do","does","must","be","can","which","that","this",
        "with","from","you","your","it","as","by","up","per","over","when","if","should"}

def _terms(s):
    return [t for t in re.findall(r"[a-z0-9]+", s.lower()) if t not in STOP]

class FakeKnowledgeBase:
    """Mirrors bedrock-agent-runtime.retrieve() — IDF-scored passages + a relevance floor."""
    def __init__(self, corpus, threshold=0.18):
        self.corpus, self.threshold = corpus, threshold
        docs = [set(_terms(txt)) for _, txt in corpus]
        N = len(docs)
        df = Counter(t for d in docs for t in d)               # document frequency
        self.idf = {t: math.log((N + 1) / (df[t] + 1)) + 1 for t in df}
        self.doc_terms = [(cid, txt, set(_terms(txt))) for cid, txt in corpus]

    def retrieve(self, query, k=3):
        q = [t for t in _terms(query)]
        q_weight = sum(self.idf.get(t, 1.0) for t in set(q)) or 1.0
        scored = []
        for cid, txt, terms in self.doc_terms:
            shared = set(q) & terms
            score = sum(self.idf.get(t, 0) for t in shared) / q_weight   # coverage of query mass
            if score >= self.threshold:
                scored.append({"id": cid, "text": txt, "score": round(score, 2)})
        scored.sort(key=lambda r: r["score"], reverse=True)
        return scored[:k]

kb = FakeKnowledgeBase(POLICY_CORPUS)
for r in kb.retrieve("What is the maximum LTV for a buy-to-let mortgage?"):
    print(f'{r["id"]}  score={r["score"]:.2f}  {r["text"][:55]}')
print("off-topic ->", kb.retrieve("What are tomorrow's lottery numbers?"))

POL-003  score=0.82  Buy-to-let mortgages require a minimum deposit of 25%; 
POL-002  score=0.55  The maximum loan-to-value (LTV) for a residential mortg
POL-013  score=0.36  First-time buyers may qualify for a 40-year maximum mor
off-topic -> []


☝ `buy-to-let` (rare) pulls **POL-003** to the top over the common-word residential chunk, and an off-topic query clears the threshold on **nothing** → `[]`. That empty list is what forces an honest refusal downstream. Same `[{id, text, score}]` contract as `bedrock-agent-runtime.retrieve()`.

## 2. `FakeGuardrail` — denied topics + PII (Day 16, condensed)

Day 16's rule: apply the guardrail on **input and output**. On the **input** we block **denied topics** — a loan agent must not give investment advice, no matter how it's asked. On the **output** we mask **PII**.

In [3]:
class FakeGuardrail:
    """Mirrors bedrock-runtime.apply_guardrail(source="INPUT"/"OUTPUT")."""
    DENIED = ["invest", "stock", "shares", "crypto", "guaranteed return", "which fund"]
    PII = re.compile(r"\b\d{8,16}\b")           # sort-code / account / card-ish digit runs

    def apply(self, text, source):
        low = text.lower()
        if source == "INPUT":
            for term in self.DENIED:
                if term in low:
                    return {"action": "BLOCKED",
                            "text": "I can only answer questions about our loan and mortgage policy."}
            return {"action": "NONE", "text": text}
        masked = self.PII.sub("[PII]", text)           # OUTPUT
        return {"action": "MASKED" if masked != text else "NONE", "text": masked}

gr = FakeGuardrail()
print(gr.apply("Should I invest my deposit in stocks?", "INPUT"))
print(gr.apply("What is the max LTV?", "INPUT"))
print(gr.apply("Your account 12345678 qualifies.", "OUTPUT"))

{'action': 'BLOCKED', 'text': 'I can only answer questions about our loan and mortgage policy.'}
{'action': 'NONE', 'text': 'What is the max LTV?'}
{'action': 'MASKED', 'text': 'Your account [PII] qualifies.'}


☝ Denied topics are caught at the **edge** — the model is never even asked. PII is masked on the way out. `apply(text, source)` mirrors `apply_guardrail(source=...)`, the same call you configured on Day 16.

## 3. `FakeBedrockRuntime.converse` — grounded generation

The model's contract: answer **using only the retrieved passages**, cite their ids, and **refuse when the passages are empty** (the anti-hallucination rule). The fake is deterministic — it grounds the answer in the top passage and cites the retrieved ids — but enforces *no context, no answer*.

In [4]:
import json

class FakeBedrockRuntime:
    """Mirrors bedrock-runtime.converse() driven by a strict RAG system prompt."""
    def converse(self, query, passages):
        if not passages:
            body = {"answer": "I cannot find this in the loan policy.",
                    "citations": [], "confidence": 0.0}
        else:
            top = passages[0]
            body = {"answer": top["text"],                       # grounded in retrieved text
                    "citations": [p["id"] for p in passages],
                    "confidence": round(min(0.99, 0.6 + top["score"]), 2)}
        return "```json\n" + json.dumps(body) + "\n```"        # models emit fenced JSON text

llm = FakeBedrockRuntime()
print(llm.converse("buy-to-let LTV?", kb.retrieve("buy-to-let mortgage maximum LTV")))
print(llm.converse("capital of France?", kb.retrieve("what is the capital of France")))

```json
{"answer": "Buy-to-let mortgages require a minimum deposit of 25%; the maximum LTV is 75%.", "citations": ["POL-003", "POL-002", "POL-013"], "confidence": 0.99}
```
```json
{"answer": "I cannot find this in the loan policy.", "citations": [], "confidence": 0.0}
```


☝ Grounded-or-refuse: with passages it answers and cites; with none it refuses at `confidence 0.0`. It emits **fenced JSON** — exactly the messy shape this morning's `JsonOutputParser` is built to eat.

## 4. Wire it as a real LangGraph `StateGraph` with conditional routing

Nodes: `guard_in → retrieve → generate → guard_out`. A **conditional edge** after `guard_in` short-circuits a blocked question straight to the end — the model and Knowledge Base are never touched. This is the deployable graph shape (AgentCore Runtime).

In [5]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.output_parsers import JsonOutputParser

verdict_parser = JsonOutputParser()      # answer text -> dict (tolerates the code fence)

class RAGState(TypedDict):
    query: str
    passages: list
    raw: str
    answer: dict
    action: str

def guard_in_node(s: RAGState):
    check = gr.apply(s["query"], "INPUT")
    if check["action"] == "BLOCKED":
        return {"answer": {"answer": check["text"], "citations": [], "confidence": 0.0},
                "action": "BLOCKED"}
    return {"action": "NONE"}

def retrieve_node(s: RAGState):
    return {"passages": kb.retrieve(s["query"], k=3)}

def generate_node(s: RAGState):
    return {"raw": llm.converse(s["query"], s["passages"])}

def guard_out_node(s: RAGState):
    verdict = verdict_parser.parse(s["raw"])          # this morning's parser
    checked = gr.apply(verdict["answer"], "OUTPUT")
    verdict["answer"] = checked["text"]
    return {"answer": verdict, "action": checked["action"]}

def route(s: RAGState):
    return END if s["action"] == "BLOCKED" else "retrieve"

b = StateGraph(RAGState)
for name, fn in [("guard_in", guard_in_node), ("retrieve", retrieve_node),
                 ("generate", generate_node), ("guard_out", guard_out_node)]:
    b.add_node(name, fn)
b.add_edge(START, "guard_in")
b.add_conditional_edges("guard_in", route, {"retrieve": "retrieve", END: END})
b.add_edge("retrieve", "generate")
b.add_edge("generate", "guard_out")
b.add_edge("guard_out", END)
agent = b.compile()
print("nodes:", list(agent.get_graph().nodes.keys()))

nodes: ['__start__', 'guard_in', 'retrieve', 'generate', 'guard_out', '__end__']


☝ A genuine `StateGraph`. `add_conditional_edges` sends a blocked query to `END` before retrieval — that's a denied topic refused *without* spending a model call. Swap the three fakes for real Bedrock clients and this exact graph ships.

In [6]:
def ask(question):
    out = agent.invoke({"query": question})
    a = out["answer"]
    print(f"Q: {question}")
    print(f"A: {a['answer']}")
    print(f"   cites={a['citations']}  confidence={a['confidence']}  guardrail={out['action']}\n")

ask("What is the maximum LTV for a buy-to-let mortgage?")   # grounded + cited
ask("How old must I be to take a personal loan?")           # grounded + cited
ask("Should I invest my deposit in the stock market?")      # denied topic -> blocked at input
ask("Who won the football match last night?")               # off-topic -> refuses

Q: What is the maximum LTV for a buy-to-let mortgage?
A: Buy-to-let mortgages require a minimum deposit of 25%; the maximum LTV is 75%.
   cites=['POL-003', 'POL-002', 'POL-013']  confidence=0.99  guardrail=NONE

Q: How old must I be to take a personal loan?
A: Applicants must be at least 18 years old to hold a personal loan.
   cites=['POL-004', 'POL-008', 'POL-005']  confidence=0.99  guardrail=NONE

Q: Should I invest my deposit in the stock market?
A: I can only answer questions about our loan and mortgage policy.
   cites=[]  confidence=0.0  guardrail=BLOCKED

Q: Who won the football match last night?
A: I cannot find this in the loan policy.
   cites=[]  confidence=0.0  guardrail=NONE



☝ Q1–2 answer with citations; the investment question is **blocked at the input edge**; the off-topic question **refuses** (`citations=[]`, `confidence=0.0`) rather than inventing. Those last two behaviours are what the eval measures next.

## 5. Evaluate — 10 questions, measure the hallucination rate

The metric a bank cares about: how often does the agent **make something up**? Keyless, we score each response as:

- **grounded** = an expected keyword appears in a **cited** answer, **or** the agent correctly **refused** an unanswerable question.
- **hallucination** = anything else — asserted a fact that isn't supported, or answered when it should have refused.

In [7]:
TEST_SET = [
    # (question, expected_keyword)   expected=None => correct behaviour is to refuse
    ("What is the maximum LTV for a residential mortgage?",        "95%"),
    ("What deposit does a buy-to-let mortgage need?",             "25%"),
    ("What is the minimum age for a personal loan?",             "18"),
    ("What is the maximum DTI ratio for approval?",              "45%"),
    ("How long must a business have traded to get a loan?",       "24 months"),
    ("What happens if a credit score is below 580?",             "manual underwriting"),
    ("How much can a standard overdraft reach?",                 "2,000"),
    ("What is the maximum mortgage term for first-time buyers?",  "40-year"),
    ("Who won the football match last night?",                    None),   # not in policy
    ("What are tomorrow's winning lottery numbers?",             None),   # not in policy
]
REFUSAL = "cannot find"

def evaluate(test_set):
    rows, halluc = [], 0
    for q, expected in test_set:
        a = agent.invoke({"query": q})["answer"]
        ans, cites = a["answer"].lower(), a["citations"]
        grounded = (REFUSAL in ans) if expected is None else (expected.lower() in ans and bool(cites))
        halluc += (not grounded)
        rows.append((q[:44], "ok" if grounded else "HALLUCINATION", a["confidence"]))
    return rows, halluc / len(test_set)

rows, rate = evaluate(TEST_SET)
for q, verdict, conf in rows:
    print(f'{verdict:14} conf={conf:<4} {q}')
print(f"\nHallucination rate: {rate:.0%}  ({int(round(rate*len(TEST_SET)))}/{len(TEST_SET)})")

ok             conf=0.99 What is the maximum LTV for a residential mo
ok             conf=0.99 What deposit does a buy-to-let mortgage need
ok             conf=0.99 What is the minimum age for a personal loan?
ok             conf=0.99 What is the maximum DTI ratio for approval?
ok             conf=0.94 How long must a business have traded to get 
ok             conf=0.99 What happens if a credit score is below 580?
ok             conf=0.99 How much can a standard overdraft reach?
ok             conf=0.99 What is the maximum mortgage term for first-
ok             conf=0.0  Who won the football match last night?
ok             conf=0.0  What are tomorrow's winning lottery numbers?

Hallucination rate: 0%  (0/10)


☝ An automated eval harness — what you'd run in **LangSmith** or with **RAGAS** on a golden set at every prompt/model change. The two unanswerable questions verify the agent *refuses* rather than confabulates; the eight factual ones test grounded recall. A non-zero rate would **gate the release**.

## 6. How this maps to real Bedrock

| You built | Real Bedrock |
|---|---|
| `POLICY_CORPUS` list | PDFs in **S3**, ingested + chunked into a **Knowledge Base** |
| `FakeKnowledgeBase.retrieve` | `bedrock-agent-runtime.retrieve(knowledgeBaseId=...)` (vector search) |
| IDF term score + threshold | **embedding cosine similarity** + score cutoff in OpenSearch (Day 20) |
| `FakeBedrockRuntime.converse` | `bedrock-runtime.converse()` / `retrieve_and_generate` with a grounding prompt |
| fenced-JSON output | Claude's actual output; `JsonOutputParser` handles it |
| `FakeGuardrail.apply(source=)` | `bedrock-runtime.apply_guardrail(source="INPUT"/"OUTPUT")` |
| conditional `StateGraph` | deploy to **AgentCore Runtime**; nodes call real clients |
| `evaluate()` harness | **LangSmith** eval / **RAGAS** faithfulness on a golden set |

```python
# real shape (needs AWS access) — same graph, real clients
import boto3
kb_rt = boto3.client("bedrock-agent-runtime", region_name="eu-west-2")
resp = kb_rt.retrieve_and_generate(
    input={"text": question},
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            "knowledgeBaseId": "LOANPOLICY01",
            "modelArn": "anthropic.claude-3-5-sonnet-20241022-v2:0",
        }})
answer  = resp["output"]["text"]
sources = [c["retrievedReferences"] for c in resp["citations"]]
```

## 7. Recap — a production-shaped RAG agent, evaluated

| Piece | Role |
|---|---|
| **Knowledge Base retrieve** | IDF-scored top-k + relevance floor — the only allowed ground truth |
| **input guardrail** | denied topics refused at the edge, before a model call |
| **grounded generation** | answer *from passages*, cite ids, refuse when empty |
| **JSON parse** (this morning) | model text → typed `{answer, citations, confidence}` |
| **output guardrail** | mask PII before the customer sees it |
| **LangGraph `StateGraph`** | guard-in → retrieve → generate → guard-out, one deployable graph |
| **eval harness** | 10 questions → **hallucination rate**, the release gate |


